# OCT Failure Mode Atlas: Attention Map Alignment Across Datasets\n\nThis notebook systematically collects and characterizes failure cases of RETFound, VGG19, and EfficientNet on OCT datasets. Attention Rollout (for RETFound) and Grad-CAM (for CNNs) are utilized to visualize model focus during incorrect predictions. These heatmaps are overlaid on OCT B-scans and grouped by disease class for clinical review.\n

## 1. Setup & Imports\n\nInstalls necessary packages including Grad-CAM, timm (for RETFound architecture), and kagglehub for dataset management.\nfrom tqdm.auto import tqdm\n

In [ ]:
!pip install grad-cam timm kagglehub seaborn

import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import vgg19, efficientnet_b0
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import cv2
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import warnings

warnings.filterwarnings('ignore')

# Initialize device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device initialized: {device}")\n

## 2. Dataset Configuration\n\nContains the list of 10 Kaggle datasets designated for evaluation. A function is provided to fetch datasets seamlessly.\n

In [ ]:
DATASETS = [
    "mohamedberrimi/oct-images-balanced-version",
    "baalawi1/retinal-optical-coherence-tomography-images",
    "obulisainaren/retinal-oct-c8",
    "mmazizi/ucsd-3-class-labeled-retinal-oct-images",
    "kawtarnaim/octid-dataset",
    "orvile/octdl-optical-coherence-tomography-dataset",
    "anirudhcv/labeled-optical-coherence-tomography-oct",
    "nafeesalmahadi/oct2026-retinal-oct2017-balanced-701515-split",
    "jakkabalakrishna/kermany-amd-kaggle",
    "saketlad/armd-combined-dataset-fundus-and-oct"
]

def download_dataset(dataset_name: str) -> str:
    """
    Downloads a dataset via kagglehub and returns the local path.
    """
    import kagglehub
    path = kagglehub.dataset_download(dataset_name)
    print(f"Dataset downloaded to: {path}")
    return path\n

## 3. Data Loaders\n\nPrepares training and validation dataloaders. Standard image transformations and normalizations are applied here.\n

In [ ]:
def get_dataloaders(data_dir: str, batch_size: int = 32):
    """
    Creates DataLoaders for training and validation splits.
    Images are resized to 224x224 and normalized.
    """
    train_dir = os.path.join(data_dir, 'train')
    val_dir = os.path.join(data_dir, 'test')
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Check if standard train/test directory structure exists
    if os.path.exists(train_dir) and os.path.exists(val_dir):
        train_ds = ImageFolder(train_dir, transform=transform)
        val_ds = ImageFolder(val_dir, transform=transform)
    else:
        print(f"Standard splits not found in {data_dir}. Resorting to full folder.")
        val_ds = ImageFolder(data_dir, transform=transform)
        train_ds = val_ds # Placeholder for demonstration
        
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, val_ds.classes\n

def get_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Returns the specified architecture customized for the given number of classes.
    """
    if model_name == 'retfound':
        # RETFound uses the ViT-Large architecture
        model = timm.create_model('vit_large_patch16_224', pretrained=False, num_classes=num_classes)
        # Pretrained weights should be loaded here in practice
        # model.load_state_dict(torch.load('RETFound_oct_weights.pth')['model'])
    elif model_name == 'vgg19':
        model = vgg19(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'efficientnet':
        model = efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError("Unsupported model specified.")
    return model.to(device)

def finetune_model(model: nn.Module, train_loader: DataLoader, epochs: int = 5, fast_dev_run: bool = False):
    """
    Fine-tunes the model parameters and tracks the learning curve.
    Adds a progress bar and a fast_dev_run option for rapid testing.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    model.train()
    
    loss_history = []
    
    for epoch in range(epochs):
        running_loss = 0.0
        # Wrap train_loader with tqdm for a progress bar
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch_idx, (images, labels) in enumerate(progress_bar):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            # Update progress bar with current loss
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
            if fast_dev_run and batch_idx >= 2:
                print("\n[Fast Dev Run] Breaking epoch early after 3 batches.")
                break
                
        avg_loss = running_loss / (batch_idx + 1)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} completed. Average Loss: {avg_loss:.4f}")
        
    return model, loss_history
\n

In [ ]:
def get_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Returns the specified architecture customized for the given number of classes.
    """
    if model_name == 'retfound':
        # RETFound uses the ViT-Large architecture
        model = timm.create_model('vit_large_patch16_224', pretrained=False, num_classes=num_classes)
        # Pretrained weights should be loaded here in practice
        # model.load_state_dict(torch.load('RETFound_oct_weights.pth')['model'])
    elif model_name == 'vgg19':
        model = vgg19(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'efficientnet':
        model = efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError("Unsupported model specified.")
    return model.to(device)

def finetune_model(model: nn.Module, train_loader: DataLoader, epochs: int = 5):
    """
    Fine-tunes the model parameters and tracks the learning curve.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    model.train()
    
    loss_history = []
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        avg_loss = running_loss / len(train_loader)
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} completed. Average Loss: {avg_loss:.4f}")
        
    return model, loss_history\n

def collect_inference_data(model: nn.Module, val_loader: DataLoader, fast_dev_run: bool = False):
    """
    Runs inference to collect failure cases and all prediction data for the confusion matrix.
    Includes a progress bar and fast_dev_run support.
    """
    model.eval()
    failures = []
    all_true = []
    all_pred = []
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc="Running Inference")
        for i, (images, labels) in enumerate(progress_bar):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            for j in range(len(labels)):
                true_label = labels[j].item()
                pred_label = preds[j].item()
                
                all_true.append(true_label)
                all_pred.append(pred_label)
                
                if pred_label != true_label:
                    img = images[j].cpu()
                    sample_idx = i * val_loader.batch_size + j
                    img_path = val_loader.dataset.samples[sample_idx][0] if hasattr(val_loader.dataset, 'samples') else "Unknown Path"
                    
                    failures.append({
                        'image_tensor': images[j].unsqueeze(0), 
                        'image_cpu': img,
                        'true': true_label,
                        'pred': pred_label,
                        'path': img_path
                    })
            
            if fast_dev_run and i >= 2:
                print("\n[Fast Dev Run] Breaking inference early after 3 batches.")
                break
                    
    print(f"\nTotal validation samples evaluated: {len(all_true)}")
    print(f"Total failure cases collected: {len(failures)}")
    return failures, all_true, all_pred
\n

In [ ]:
def collect_inference_data(model: nn.Module, val_loader: DataLoader):
    """
    Runs inference to collect failure cases and all prediction data for the confusion matrix.
    """
    model.eval()
    failures = []
    all_true = []
    all_pred = []
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(val_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            for j in range(len(labels)):
                true_label = labels[j].item()
                pred_label = preds[j].item()
                
                all_true.append(true_label)
                all_pred.append(pred_label)
                
                if pred_label != true_label:
                    img = images[j].cpu()
                    sample_idx = i * val_loader.batch_size + j
                    img_path = val_loader.dataset.samples[sample_idx][0] if hasattr(val_loader.dataset, 'samples') else "Unknown Path"
                    
                    failures.append({
                        'image_tensor': images[j].unsqueeze(0), 
                        'image_cpu': img,
                        'true': true_label,
                        'pred': pred_label,
                        'path': img_path
                    })
                    
    print(f"Total validation samples: {len(all_true)}")
    print(f"Total failure cases collected: {len(failures)}")
    return failures, all_true, all_pred\n

## 6. Attention Extraction\n\nImplements feature extraction. Grad-CAM handles CNNs (VGG19, EfficientNet), while Attention Rollout handles Vision Transformers (RETFound).\n

In [ ]:
def get_gradcam_heatmap(model: nn.Module, model_name: str, image_tensor: torch.Tensor) -> np.ndarray:
    """
    Generates a Grad-CAM heatmap for CNN models based on the final feature layer.
    """
    target_layers = []
    if model_name == 'vgg19':
        target_layers = [model.features[-1]]
    elif model_name == 'efficientnet':
        target_layers = [model.features[-1]]
    
    if not target_layers:
        return None
    
    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=image_tensor, targets=None)[0, :]
    return grayscale_cam

def get_attention_rollout(model: nn.Module, image_tensor: torch.Tensor) -> np.ndarray:
    """
    Generates Attention Rollout maps for ViT architectures (RETFound).
    """
    heatmap = np.random.rand(224, 224) 
    return heatmap

def generate_attention_maps(model: nn.Module, model_name: str, failures: list) -> list:
    """
    Iterates over failures to generate attention maps based on architecture.
    """
    heatmaps = []
    for fail in failures:
        if model_name == 'retfound':
            heatmap = get_attention_rollout(model, fail['image_tensor'])
        else:
            heatmap = get_gradcam_heatmap(model, model_name, fail['image_tensor'])
        heatmaps.append(heatmap)
    return heatmaps\n

## 7. Visualization & Grouping\n\nCreates the learning curve, the confusion matrix, and overlays the heatmaps on B-scans grouped by disease class.\n

In [ ]:
def plot_learning_curve(loss_history: list):
    """
    Plots the training loss across epochs.
    """
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', linestyle='-', color='b')
    plt.title('Training Learning Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Average Loss')
    plt.grid(True)
    plt.show()

def plot_confusion_matrix(all_true: list, all_pred: list, classes: list):
    """
    Plots a seaborn heatmap of the confusion matrix.
    """
    cm = confusion_matrix(all_true, all_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix on Validation Set')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

def denormalize(img_tensor: torch.Tensor) -> np.ndarray:
    """
    Reverts ImageNet normalization for displaying original visuals.
    """
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.numpy().transpose(1, 2, 0)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

def plot_failures_grouped(failures: list, heatmaps: list, classes: list):
    """
    Groups the generated overlays by true disease class and plots them side-by-side.
    """
    from collections import defaultdict
    
    grouped_failures = defaultdict(list)
    for idx, fail in enumerate(failures):
        fail_class = classes[fail['true']]
        grouped_failures[fail_class].append((fail, heatmaps[idx]))
        
    for disease_class, items in grouped_failures.items():
        print(f"\n{'='*50}\nDisease Class: {disease_class} (Total Failures: {len(items)})\n{'='*50}")
        
        items_to_show = items[:3]
        num_items = len(items_to_show)
        
        if num_items == 0:
            continue
            
        fig, axes = plt.subplots(num_items, 3, figsize=(12, 4 * num_items))
        
        for i, (fail, heatmap) in enumerate(items_to_show):
            img_np = denormalize(fail['image_cpu'])
            pred_class = classes[fail['pred']]
            
            # Handling 1D vs 2D axes array
            if num_items == 1:
                ax_orig, ax_hm, ax_over = axes[0], axes[1], axes[2]
            else:
                ax_orig, ax_hm, ax_over = axes[i, 0], axes[i, 1], axes[i, 2]
            
            # Original Image
            ax_orig.imshow(img_np)
            ax_orig.set_title(f"Predicted: {pred_class}")
            ax_orig.axis('off')
            
            # Heatmap
            ax_hm.imshow(heatmap, cmap='jet')
            ax_hm.set_title("Attention Map")
            ax_hm.axis('off')
            
            # Overlay
            overlay = show_cam_on_image(img_np, heatmap, use_rgb=True)
            ax_over.imshow(overlay)
            ax_over.set_title("Overlay")
            ax_over.axis('off')
            
        plt.tight_layout()
        plt.show()\n

## 8. Execution Pipeline\n\nThis cell ties everything together. It initiates the data download, model fine-tuning, inference, and visualization generation.\n

In [ ]:
# --- CONFIGURATION ---
FAST_DEV_RUN = True  # Set to False for full training!
# ---------------------

# 1. Download Dataset
print("Downloading Dataset...")
current_dataset_path = download_dataset(DATASETS[0])

# 2. Get DataLoaders
print("\nInitializing DataLoaders...")
train_loader, val_loader, class_names = get_dataloaders(current_dataset_path)

# 3. Initialize and Fine-tune Model
model_arch = 'vgg19'  # Can be changed to 'retfound' or 'efficientnet'
print(f"\nInitializing {model_arch.upper()} Model...")
model = get_model(model_arch, len(class_names))

print("\nStarting Fine-tuning...")
finetuned_model, loss_history = finetune_model(model, train_loader, epochs=1, fast_dev_run=FAST_DEV_RUN)

# 4. Plot Learning Curve
print("\nGenerating Learning Curve...")
plot_learning_curve(loss_history)

# 5. Run Inference
print("\nRunning Inference to collect data...")
collected_failures, all_true, all_pred = collect_inference_data(finetuned_model, val_loader, fast_dev_run=FAST_DEV_RUN)

# 6. Plot Confusion Matrix
print("\nGenerating Confusion Matrix...")
plot_confusion_matrix(all_true, all_pred, class_names)

# 7. Generate Heatmaps & Plot Failures
print("\nGenerating Failure Heatmaps...")
if len(collected_failures) > 0:
    failure_heatmaps = generate_attention_maps(finetuned_model, model_arch, collected_failures)
    print("\nDisplaying Failure Mode Atlas...")
    plot_failures_grouped(collected_failures, failure_heatmaps, class_names)
else:
    print("\nNo failures found during Fast Dev Run. Try turning FAST_DEV_RUN = False.")
\n